# Explicación Detallada de la Práctica 3
### Métodos de Generación de Números Pseudoaleatorios: Cuadrado Medio y Congruencial Mixto

Este cuaderno de Jupyter explica paso a paso el funcionamiento y la ejecución de los algoritmos implementados en `Practica3.py`.
A través de celdas de código interactivas, analizaremos cómo se generan los números pseudoaleatorios y cómo se detectan sus ciclos y periodos.


## 1. Método del Cuadrado Medio (Mid-Square Method)

El método del cuadrado medio es uno de los primeros algoritmos propuestos para generar números pseudoaleatorios (por John von Neumann en 1949).

### Algoritmo paso a paso:
1. **Semilla Inicial**: Se comienza con un número de $n$ dígitos (típicamente $n$ es par, por ejemplo, $n=4$).
2. **Elevación al Cuadrado**: Se calcula el cuadrado del número actual ($X_i^2$).
3. **Relleno de Ceros (Padding)**: Si el resultado tiene menos de $2n$ dígitos, se añaden ceros a la izquierda para garantizar una longitud exacta de $2n$ dígitos.
4. **Extracción**: Se seleccionan los $n$ dígitos centrales del número obtenido. Este valor será el siguiente número de la secuencia ($X_{i+1}$).
5. **Repetición**: Se utiliza el número extraído como la nueva semilla y se repite el proceso hasta que ocurra una repetición (un número ya generado vuelve a aparecer, indicando que el generador ha entrado en un ciclo).


In [1]:
def cuadrado_medio_hasta_repetir(semilla_inicial):
    secuencia = []
    repetidos = set()
    semilla = semilla_inicial
    n = len(str(semilla_inicial))  # Cantidad de dígitos (por ejemplo, n=4)
    
    pos_repeticion = -1
    valor_repetido = None
    iteracion = 1

    while True:
        al_cuadrado = semilla ** 2  # Elevar la semilla al cuadrado
        # Ajustar a longitud 2n (8 dígitos) agregando ceros a la izquierda si es necesario
        cadena_cuadrado = str(al_cuadrado).zfill(2 * n)
        
        # Extraer los n dígitos centrales
        inicio = (len(cadena_cuadrado) - n) // 2
        centro = int(cadena_cuadrado[inicio:inicio + n])
        
        secuencia.append(centro)
        
        # Comprobar si el número generado ya existía en la secuencia
        if centro in repetidos:
            pos_repeticion = iteracion
            valor_repetido = centro
            break
            
        repetidos.add(centro)
        semilla = centro
        iteracion += 1
        
    return secuencia, pos_repeticion, valor_repetido

def imprimir_bloque_justificado(secuencia):
    # Imprime una lista en bloques de 10 números alineados en columnas de ancho 7
    for i in range(0, len(secuencia), 10):
        bloque = secuencia[i:i+10]
        linea_formateada = "".join(f"{num:<7}" for num in bloque)
        print(linea_formateada)


### Ejecución del Método del Cuadrado Medio
Utilizaremos la semilla de la práctica, **7326**, y observaremos cuántas iteraciones toma el método antes de entrar en un ciclo repetitivo.


In [2]:
semilla_inicial_cm = 7326
secuencia_cm, pos_cm, val_cm = cuadrado_medio_hasta_repetir(semilla_inicial_cm)

print("="*80)
print("MÉTODO DEL CUADRADO MEDIO")
print("="*80)
print(f"Semilla inicial: {semilla_inicial_cm}")
print(f"La repetición ocurre en la iteración N°: {pos_cm} (Valor repetido: {val_cm})\n")
print(f"Secuencia generada completa (Iteraciones 1 a {pos_cm}):")
print("-" * 80)
imprimir_bloque_justificado(secuencia_cm)
print("-" * 80)


MÉTODO DEL CUADRADO MEDIO
Semilla inicial: 7326
La repetición ocurre en la iteración N°: 55 (Valor repetido: 0)

Secuencia generada completa (Iteraciones 1 a 55):
--------------------------------------------------------------------------------
6702   9168   522    2724   4201   6484   422    1780   1684   8358   
8561   2907   4506   3040   2416   8370   569    3237   4781   8579   
5992   9040   7216   706    4984   8402   5936   2360   5696   4444   
7491   1150   3225   4006   480    2304   3084   5110   1121   2566   
5843   1406   9768   4138   1230   5129   3066   4003   240    576    
3317   24     5      0      0      
--------------------------------------------------------------------------------


## 2. Método Congruencial Mixto con Rastreo de Ciclos

El método congruencial mixto es uno de los algoritmos más populares y estudiados. Se basa en la teoría de números y utiliza aritmética modular.

### Ecuación de Recurrencia:
$$X_{i} = (a \cdot X_{i-1} + b) \pmod M$$

Donde:
* $X_{i}$ es el número pseudoaleatorio generado en el paso $i$.
* $X_0$ es la semilla inicial ($X_0 \ge 0$).
* $a$ es la constante multiplicativa o multiplicador ($a > 0$).
* $b$ es la constante aditiva o incremento ($b > 0$).
* $M$ es el módulo ($M > 0$).

### Periodo y Rastreo de Ciclos:
Como los números generados están contenidos en el espacio modular $[0, M-1]$, el generador eventualmente repetirá un valor. Una vez que se repite un valor, toda la secuencia posterior se repetirá exactamente igual, formando un bucle o ciclo.

El **periodo** es el número de elementos únicos generados antes de que comience el ciclo. Para rastrearlo:
1. Guardamos cada número generado en un diccionario junto con la iteración en la que apareció: `{ valor: iteracion }`.
2. Si un valor generado ya existe en el diccionario, hemos detectado el fin del periodo.
3. El periodo exacto es la diferencia: $\text{Periodo} = \text{iteración segunda aparición} - \text{iteración primera aparición}$.


In [3]:
def congruencial_rastreo_ciclos(semilla_inicial, a, b, M, max_iteraciones=10000):
    # Diccionario para registrar { valor_generado: numero_de_iteracion_donde_aparecio }
    registro_apariciones = {}
    secuencia_historica = []
    
    # Ajustar la semilla al espacio modular por seguridad
    X = semilla_inicial % M
    
    pos_primera = -1
    pos_segunda = -1
    valor_repetido = None

    for i in range(1, max_iteraciones + 1):
        # Fórmula congruencial mixta: X_i = (a * X_{i-1} + b) % M
        X = (a * X + b) % M
        secuencia_historica.append(X)
        
        # Comprobar si ya se había generado este valor
        if X in registro_apariciones:
            pos_primera = registro_apariciones[X]
            pos_segunda = i
            valor_repetido = X
            break
            
        registro_apariciones[X] = i
        
    return secuencia_historica, pos_primera, pos_segunda, valor_repetido


### Ejecución de los Ejercicios de la Práctica
Evaluaremos los parámetros de la práctica para analizar la longitud del periodo bajo diferentes semillas y configuraciones:

* **Ejercicio 3**: $a=13, b=7, M=1024$ con semillas $[473, 8432, 4728]$
* **Ejercicio 4**: $a=25, b=13, M=2048$ con semillas $[2537, 4694, 6598]$
* **Ejercicio 5**: $a=45, b=23, M=512$ con semilla $[9825]$


In [4]:
ejercicios_congruenciales = [
    {"num": "3", "a": 13, "b": 7, "M": 1024, "semillas": [473, 8432, 4728]},
    {"num": "4", "a": 25, "b": 13, "M": 2048, "semillas": [2537, 4694, 6598]},
    {"num": "5", "a": 45, "b": 23, "M": 512, "semillas": [9825]}
]

for ej in ejercicios_congruenciales:
    print("\n" + "="*80)
    print(f"EJERCICIO {ej['num']}: Método Congruencial (a={ej['a']}, b={ej['b']}, M={ej['M']})")
    print("="*80)
    
    for s in ej["semillas"]:
        secuencia, p_primera, p_segunda, val = congruencial_rastreo_ciclos(s, ej["a"], ej["b"], ej["M"])
        lapso_periodo = p_segunda - p_primera
        
        print(f"-> Semilla {s} detonó ciclo en la iteración N°: {p_segunda} (Valor repetido: {val})")
        print(f"   Historial matemático: Apareció por 1° vez en iteración {p_primera} y por 2° vez en {p_segunda}.")
        print(f"   El periodo exacto de este ciclo es de: {lapso_periodo} iteraciones.\n")
        
        # --- BLOQUE 1: Ventana de la primera aparición ---
        idx_inicio_p1 = max(0, p_primera - 3)
        idx_fin_p1 = min(len(secuencia), p_primera + 2)
        bloque_primera = secuencia[idx_inicio_p1:idx_fin_p1]
        str_bloque_p1 = "  ".join(f"[{num}]" if (idx_inicio_p1 + idx + 1) == p_primera else f"{num}" for idx, num in enumerate(bloque_primera))
        
        # --- BLOQUE 2: Ventana de la segunda aparición ---
        idx_inicio_p2 = max(0, p_segunda - 5)
        bloque_segunda = secuencia[idx_inicio_p2:p_segunda]
        str_bloque_p2 = "  ".join(f"[{num}]" if (idx_inicio_p2 + idx + 1) == p_segunda else f"{num}" for idx, num in enumerate(bloque_segunda))
        
        # Demostración del entorno de las repeticiones
        print(f"   [Muestra Visual del Lapso]")
        print(f"   Entorno de la 1° aparición (Iteraciones {idx_inicio_p1+1} a {idx_fin_p1}):")
        print(f"     {str_bloque_p1}")
        print(f"     ...")
        print(f"   Entorno de la 2° aparición e intercepción (Iteraciones {idx_inicio_p2+1} a {p_segunda}):")
        print(f"     {str_bloque_p2}")
        print("   " + "-" * 72)



EJERCICIO 3: Método Congruencial (a=13, b=7, M=1024)
-> Semilla 473 detonó ciclo en la iteración N°: 1025 (Valor repetido: 12)
   Historial matemático: Apareció por 1° vez en iteración 1 y por 2° vez en 1025.
   El periodo exacto de este ciclo es de: 1024 iteraciones.

   [Muestra Visual del Lapso]
   Entorno de la 1° aparición (Iteraciones 1 a 3):
     [12]  163  78
     ...
   Entorno de la 2° aparición e intercepción (Iteraciones 1021 a 1025):
     888  287  666  473  [12]
   ------------------------------------------------------------------------
-> Semilla 8432 detonó ciclo en la iteración N°: 1025 (Valor repetido: 55)
   Historial matemático: Apareció por 1° vez en iteración 1 y por 2° vez en 1025.
   El periodo exacto de este ciclo es de: 1024 iteraciones.

   [Muestra Visual del Lapso]
   Entorno de la 1° aparición (Iteraciones 1 a 3):
     [55]  722  177
     ...
   Entorno de la 2° aparición e intercepción (Iteraciones 1021 a 1025):
     371  734  333  240  [55]
   ---------